# ACC102 Track 4 - Stock Financial Analysis Tool
## Python Data Product - Analytical Workflow (iFinD Data Source)

**Student:** [Your Name]  
**Student ID:** [Your ID]  
**Date:** April 2025  
**Track:** 4 - Interactive Data Analysis Tool  
**Data Source:** iFinD Financial Database

## 1. Problem Definition

### Analytical Problem
Individual investors and finance students often struggle to quickly assess stock performance and understand technical indicators. Traditional financial platforms can be overwhelming with excessive information, making it difficult to focus on key metrics and trends.

### Target User/Audience
- **Primary Users:** Finance students, individual investors, and beginners in stock market analysis
- **Use Case:** Quick stock performance assessment, learning technical analysis concepts, and making informed investment decisions
- **Value Proposition:** An accessible, interactive tool that simplifies complex financial data into actionable insights

## 2. Data Source and Description

### Data Source
- **Source:** iFinD Financial Database (Academic Financial Data Platform)
- **Access Date:** April 16, 2025
- **Data Type:** Historical stock price data (OHLCV - Open, High, Low, Close, Volume)
- **Coverage:** Global markets including US stocks, Hong Kong stocks, and China A-shares

### Dataset Characteristics
- **Frequency:** Daily price data
- **Fields:**
  - Open: Opening price for the trading day
  - High: Highest price during the trading day
  - Low: Lowest price during the trading day
  - Close: Closing price for the trading day
  - Volume: Number of shares traded
  - Company Info: Name, exchange, IPO date, employees, etc.

### Stock Symbol Formats
- US Stocks: `AAPL.O` (add .O suffix)
- Hong Kong Stocks: `0001.HK`
- China A-Shares: `600223.SH` (Shanghai) or `000001.SZ` (Shenzhen)

## 3. Setup and Library Imports

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For accessing iFinD data source
import sys
sys.path.append('/app/.kimi/skills')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Pandas version: 2.2.3
NumPy version: 2.4.4


## 4. Data Acquisition from iFinD

In [7]:
import pandas as pd
import numpy as np

# 定义股票代码和日期范围
stock_symbol = 'AAPL'  # WRDS 格式不需要 .O 后缀
start_date = '2024-01-01'
end_date = '2025-04-16'

print(f"Fetching data for {stock_symbol}...")
print(f"Date range: {start_date} to {end_date}")

# 修改后的函数 - 使用 WRDS
import wrds

def fetch_stock_data_wrds(username, password, symbol, start, end):
    """Fetch stock data from WRDS (CRSP database)"""
    
    # 连接 WRDS
    db = wrds.Connection(wrds_username=username, wrds_password=password)
    
    # 从 CRSP 获取日度股票数据
    query = f"""
    SELECT 
        a.date,
        a.permno,
        a.ticker,
        a.prc as close_price,
        a.openprc as open_price,
        a.bidlo as low_price,
        a.askhi as high_price,
        a.vol as volume,
        a.ret as daily_return
    FROM crsp.dsf a
    WHERE a.ticker = '{symbol}'
    AND a.date BETWEEN '{start}' AND '{end}'
    AND a.prc IS NOT NULL
    ORDER BY a.date
    """
    
    df = db.raw_sql(query)
    
    if df.empty:
        print(f"No data found for {symbol}")
        db.close()
        return None, None
    
    # 数据处理
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    # CRSP 价格调整（负值表示无收盘价，取绝对值）
    df['close_price'] = df['close_price'].abs()
    df['open_price'] = df['open_price'].abs()
    df['low_price'] = df['low_price'].abs()
    df['high_price'] = df['high_price'].abs()
    
    # 重命名列以匹配原有格式
    df.rename(columns={
        'open_price': 'Open',
        'high_price': 'High',
        'low_price': 'Low',
        'close_price': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # 获取股票信息
    info_query = f"""
    SELECT 
        comnam as company_name,
        permno,
        ticker,
        exchcd as exchange_code
    FROM crsp.dsenames
    WHERE ticker = '{symbol}'
    LIMIT 1
    """
    info_df = db.raw_sql(info_query)
    info = info_df.iloc[0].to_dict() if not info_df.empty else {}
    
    db.close()
    
    return df, info


# ==========================================
# 使用说明：
# ==========================================

# 请填入你的 WRDS 账号密码
WRDS_USERNAME = "你的WRDS用户名"  # 替换为你的用户名
WRDS_PASSWORD = "你的WRDS密码"    # 替换为你的密码

# 检查是否已填入真实信息
if WRDS_USERNAME == "你的WRDS用户名" or WRDS_PASSWORD == "你的WRDS密码":
    print("\n" + "="*60)
    print("⚠ 请先修改代码中的 WRDS 账号密码！")
    print("="*60)
    print("\n请将以下两行改为你的真实WRDS账号：")
    print('WRDS_USERNAME = "你的WRDS用户名"')
    print('WRDS_PASSWORD = "你的WRDS密码"')
else:
    # 获取数据
    df, info = fetch_stock_data_wrds(WRDS_USERNAME, WRDS_PASSWORD, 
                                     stock_symbol, start_date, end_date)
    
    if df is not None:
        print(f"\nData fetched successfully!")
        print(f"Total records: {len(df)}")
        print(f"\nStock info: {info}")
        print(f"\nFirst 5 rows:")
        print(df.head())

Fetching data for AAPL...
Date range: 2024-01-01 to 2025-04-16

⚠ 请先修改代码中的 WRDS 账号密码！

请将以下两行改为你的真实WRDS账号：
WRDS_USERNAME = "你的WRDS用户名"
WRDS_PASSWORD = "你的WRDS密码"


In [8]:
!pip install wrds -q
print("✓ wrds 安装完成")

✓ wrds 安装完成


In [16]:
# 完整修复版 - 使用 WRDS 替代 mshtools/iFinD

import pandas as pd
import numpy as np
import wrds

# 定义股票代码和日期范围
stock_symbol = 'AAPL'  # WRDS 格式不需要 .O 后缀
start_date = '2024-01-01'
end_date = '2025-04-16'

print(f"Fetching data for {stock_symbol}...")
print(f"Date range: {start_date} to {end_date}")

def fetch_stock_data_wrds(username, password, symbol, start, end):
    """Fetch stock data from WRDS (CRSP database)"""
    
    # 连接 WRDS
    db = wrds.Connection(wrds_username=username, wrds_password=password)
    
    # 从 CRSP 获取日度股票数据
    query = f"""
    SELECT 
        a.date,
        a.permno,
        a.ticker,
        a.prc as close_price,
        a.openprc as open_price,
        a.bidlo as low_price,
        a.askhi as high_price,
        a.vol as volume,
        a.ret as daily_return
    FROM crsp.dsf a
    WHERE a.ticker = '{symbol}'
    AND a.date BETWEEN '{start}' AND '{end}'
    AND a.prc IS NOT NULL
    ORDER BY a.date
    """
    
    df = db.raw_sql(query)
    
    if df.empty:
        print(f"No data found for {symbol}")
        db.close()
        return None, None
    
    # 数据处理
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    # CRSP 价格调整（负值表示无收盘价，取绝对值）
    df['close_price'] = df['close_price'].abs()
    df['open_price'] = df['open_price'].abs()
    df['low_price'] = df['low_price'].abs()
    df['high_price'] = df['high_price'].abs()
    
    # 重命名列以匹配原有格式
    df.rename(columns={
        'open_price': 'Open',
        'high_price': 'High',
        'low_price': 'Low',
        'close_price': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # 获取股票信息
    info_query = f"""
    SELECT 
        comnam as company_name,
        permno,
        ticker,
        exchcd as exchange_code
    FROM crsp.dsenames
    WHERE ticker = '{symbol}'
    LIMIT 1
    """
    info_df = db.raw_sql(info_query)
    info = info_df.iloc[0].to_dict() if not info_df.empty else {}
    
    db.close()
    
    return df, info


# ==========================================
# 填入你的 WRDS 账号密码
# ==========================================

WRDS_USERNAME = "laferrry"  # ← 替换为你的用户名
WRDS_PASSWORD = "Lfy060808!@#"    # ← 替换为你的密码

# 检查是否已填入真实信息
if WRDS_USERNAME == "laferrry" or WRDS_PASSWORD == "Lfy060808!@#":
    print("\n" + "="*60)
    print("⚠ 请先修改代码中的 WRDS 账号密码！")
    print("="*60)
    print("\n请将以下两行改为你的真实WRDS账号：")
    print(f'WRDS_USERNAME = "laferrry"  →  WRDS_USERNAME = "xxx"')
    print(f'WRDS_PASSWORD = "Lfy060808！@#
          "    →  WRDS_PASSWORD = "xxx"')
    print("\n然后重新运行代码")
else:
    # 获取数据
    df, info = fetch_stock_data_wrds(WRDS_USERNAME, WRDS_PASSWORD, 
                                     stock_symbol, start_date, end_date)
    
    if df is not None:
        print(f"\n✓ Data fetched successfully!")
        print(f"Total records: {len(df)}")
        if info:
            print(f"\nStock info:")
            for key, value in info.items():
                print(f"  {key}: {value}")
        print(f"\nFirst 5 rows:")
        print(df.head())
        print(f"\nData columns: {list(df.columns)}")

SyntaxError: unterminated f-string literal (detected at line 99) (860172646.py, line 99)

In [11]:

import pandas as pd
import numpy as np
import wrds

# 定义股票代码和日期范围
stock_symbol = 'AAPL'  # WRDS 格式不需要 .O 后缀
start_date = '2024-01-01'
end_date = '2025-04-16'

print(f"Fetching data for {stock_symbol}...")
print(f"Date range: {start_date} to {end_date}")

def fetch_stock_data_wrds(username, password, symbol, start, end):
    """Fetch stock data from WRDS (CRSP database)"""
    
    # 连接 WRDS
    db = wrds.Connection(wrds_username=username, wrds_password=password)
    
    # 从 CRSP 获取日度股票数据
    query = f"""
    SELECT 
        a.date,
        a.permno,
        a.ticker,
        a.prc as close_price,
        a.openprc as open_price,
        a.bidlo as low_price,
        a.askhi as high_price,
        a.vol as volume,
        a.ret as daily_return
    FROM crsp.dsf a
    WHERE a.ticker = '{symbol}'
    AND a.date BETWEEN '{start}' AND '{end}'
    AND a.prc IS NOT NULL
    ORDER BY a.date
    """
    
    df = db.raw_sql(query)
    
    if df.empty:
        print(f"No data found for {symbol}")
        db.close()
        return None, None
    
    # 数据处理
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    # CRSP 价格调整（负值表示无收盘价，取绝对值）
    df['close_price'] = df['close_price'].abs()
    df['open_price'] = df['open_price'].abs()
    df['low_price'] = df['low_price'].abs()
    df['high_price'] = df['high_price'].abs()
    
    # 重命名列以匹配原有格式
    df.rename(columns={
        'open_price': 'Open',
        'high_price': 'High',
        'low_price': 'Low',
        'close_price': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # 获取股票信息
    info_query = f"""
    SELECT 
        comnam as company_name,
        permno,
        ticker,
        exchcd as exchange_code
    FROM crsp.dsenames
    WHERE ticker = '{symbol}'
    LIMIT 1
    """
    info_df = db.raw_sql(info_query)
    info = info_df.iloc[0].to_dict() if not info_df.empty else {}
    
    db.close()
    
    return df, info


# ==========================================
# 填入你的 WRDS 账号密码
# ==========================================

WRDS_USERNAME = "laferrry"  # ← 替换为你的用户名
WRDS_PASSWORD = "Lfy060808!@#"    # ← 替换为你的密码

# 检查是否已填入真实信息
if WRDS_USERNAME == "你的WRDS用户名" or WRDS_PASSWORD == "你的WRDS密码":
    print("\n" + "="*60)
    print("⚠ 请先修改代码中的 WRDS 账号密码！")
    print("="*60)
    print("\n请将以下两行改为你的真实WRDS账号：")
    print(f'WRDS_USERNAME = "你的WRDS用户名"  →  WRDS_USERNAME = "xxx"')
    print(f'WRDS_PASSWORD = "你的WRDS密码"    →  WRDS_PASSWORD = "xxx"')
    print("\n然后重新运行代码")
else:
    # 获取数据
    df, info = fetch_stock_data_wrds(WRDS_USERNAME, WRDS_PASSWORD, 
                                     stock_symbol, start_date, end_date)
    
    if df is not None:
        print(f"\n✓ Data fetched successfully!")
        print(f"Total records: {len(df)}")
        if info:
            print(f"\nStock info:")
            for key, value in info.items():
                print(f"  {key}: {value}")
        print(f"\nFirst 5 rows:")
        print(df.head())
        print(f"\nData columns: {list(df.columns)}")

Fetching data for AAPL...
Date range: 2024-01-01 to 2025-04-16
Loading library list...
Done


ProgrammingError: (psycopg2.errors.UndefinedColumn) column a.ticker does not exist
LINE 5:         a.ticker,
                ^

[SQL: 
    SELECT 
        a.date,
        a.permno,
        a.ticker,
        a.prc as close_price,
        a.openprc as open_price,
        a.bidlo as low_price,
        a.askhi as high_price,
        a.vol as volume,
        a.ret as daily_return
    FROM crsp.dsf a
    WHERE a.ticker = 'AAPL'
    AND a.date BETWEEN '2024-01-01' AND '2025-04-16'
    AND a.prc IS NOT NULL
    ORDER BY a.date
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)

## 5. Company Information

In [3]:
# Display company information
print("Company Information:")
print("="*50)
if info:
    print(f"Company Name (EN): {info.get('en_short_name', 'N/A')}")
    print(f"Company Name (CN): {info.get('stock_short_name', 'N/A')}")
    print(f"Stock Code: {info.get('stock_code', 'N/A')}")
    print(f"Exchange: {info.get('listed_mkt', 'N/A')}")
    ipo_date = info.get('ipo_date', 'N/A')
    if ipo_date != 'N/A':
        ipo_date = f"{ipo_date[:4]}-{ipo_date[4:6]}-{ipo_date[6:]}"
    print(f"IPO Date: {ipo_date}")
    staff = info.get('staff_num', 'N/A')
    if staff and str(staff).isdigit():
        staff = f"{int(staff):,}"
    print(f"Employees: {staff}")
    print(f"Website: {info.get('corp_website', 'N/A')}")
else:
    print("No company information available.")

Company Information:


NameError: name 'info' is not defined

## 6. Data Cleaning and Validation

In [ ]:
# Check data structure and quality
print("Data Information:")
print("="*50)
print(f"Shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicated rows: {df.duplicated().sum()}")

In [ ]:
# Basic statistics
print("\nDescriptive Statistics:")
print("="*50)
df[['Open', 'High', 'Low', 'Close', 'Volume']].describe()

## 7. Feature Engineering - Technical Indicators

In [ ]:
# Calculate Simple Moving Average (SMA)
def calculate_sma(data, period=20):
    """Calculate Simple Moving Average"""
    return data['Close'].rolling(window=period).mean()

# Calculate Exponential Moving Average (EMA)
def calculate_ema(data, period=20):
    """Calculate Exponential Moving Average"""
    return data['Close'].ewm(span=period, adjust=False).mean()

# Calculate Relative Strength Index (RSI)
def calculate_rsi(data, period=14):
    """Calculate Relative Strength Index"""
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Calculate Daily Returns
def calculate_returns(data):
    """Calculate daily percentage returns"""
    return data['Close'].pct_change() * 100

# Calculate Volatility (Rolling Standard Deviation)
def calculate_volatility(data, period=20):
    """Calculate rolling volatility"""
    return data['Close'].rolling(window=period).std()

# Apply calculations
df['SMA_20'] = calculate_sma(df, 20)
df['EMA_20'] = calculate_ema(df, 20)
df['RSI'] = calculate_rsi(df, 14)
df['Daily_Return'] = calculate_returns(df)
df['Volatility'] = calculate_volatility(df, 20)

print("Technical indicators calculated successfully!")
print("\nNew columns added:")
print(['SMA_20', 'EMA_20', 'RSI', 'Daily_Return', 'Volatility'])

## 8. Exploratory Data Analysis (EDA)

In [ ]:
# Key performance metrics
latest_price = df['Close'].iloc[-1]
period_start_price = df['Close'].iloc[0]
period_return = ((latest_price - period_start_price) / period_start_price) * 100
highest_price = df['High'].max()
lowest_price = df['Low'].min()
avg_volume = df['Volume'].mean()
volatility_annual = df['Daily_Return'].std() * np.sqrt(252)

print("Key Performance Metrics:")
print("="*50)
print(f"Current Price: ${latest_price:.2f}")
print(f"Period Return: {period_return:.2f}%")
print(f"Period High: ${highest_price:.2f}")
print(f"Period Low: ${lowest_price:.2f}")
print(f"Average Volume: {avg_volume:,.0f}")
print(f"Annualized Volatility: {volatility_annual:.2f}%")

### 8.1 Price Trend Visualization

In [ ]:
# Create price chart with moving averages
fig = go.Figure()

# Add closing price
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Close'],
    name='Close Price',
    line=dict(color='blue', width=2)
))

# Add SMA
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['SMA_20'],
    name='SMA (20)',
    line=dict(color='orange', width=2)
))

# Add EMA
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['EMA_20'],
    name='EMA (20)',
    line=dict(color='green', width=2)
))

fig.update_layout(
    title=f'{stock_symbol} Stock Price with Moving Averages',
    xaxis_title='Date',
    yaxis_title='Price ($)',
    template='plotly_white',
    height=500
)

fig.show()

### 8.2 RSI Visualization

In [ ]:
# Create RSI chart
fig_rsi = go.Figure()

fig_rsi.add_trace(go.Scatter(
    x=df.index,
    y=df['RSI'],
    name='RSI',
    line=dict(color='purple', width=2)
))

# Add overbought/oversold lines
fig_rsi.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought")
fig_rsi.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold")

fig_rsi.update_layout(
    title='Relative Strength Index (RSI)',
    xaxis_title='Date',
    yaxis_title='RSI',
    template='plotly_white',
    height=350,
    yaxis=dict(range=[0, 100])
)

fig_rsi.show()

### 8.3 Returns Distribution

In [ ]:
# Daily returns histogram
fig_hist = px.histogram(
    df.dropna(),
    x='Daily_Return',
    nbins=50,
    title='Distribution of Daily Returns',
    labels={'Daily_Return': 'Daily Return (%)'},
    template='plotly_white'
)

fig_hist.add_vline(x=0, line_dash="dash", line_color="red")
fig_hist.update_layout(height=400)

fig_hist.show()

### 8.4 Volume Analysis

In [ ]:
# Volume chart
fig_vol = go.Figure()

fig_vol.add_trace(go.Bar(
    x=df.index,
    y=df['Volume'],
    name='Volume',
    marker_color='steelblue'
))

fig_vol.update_layout(
    title='Trading Volume Over Time',
    xaxis_title='Date',
    yaxis_title='Volume',
    template='plotly_white',
    height=350
)

fig_vol.show()

## 9. Monthly Performance Analysis

In [ ]:
# Resample to monthly data
df_monthly = df.resample('ME').agg({
    'Close': 'last',
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Volume': 'sum'
})

# Calculate monthly returns
df_monthly['Monthly_Return'] = ((df_monthly['Close'] - df_monthly['Open']) / df_monthly['Open']) * 100
df_monthly = df_monthly.dropna()

# Create monthly returns bar chart
colors = ['green' if x >= 0 else 'red' for x in df_monthly['Monthly_Return']]

fig_monthly = go.Figure()
fig_monthly.add_trace(go.Bar(
    x=df_monthly.index.strftime('%Y-%m'),
    y=df_monthly['Monthly_Return'],
    marker_color=colors,
    name='Monthly Return'
))

fig_monthly.update_layout(
    title='Monthly Returns',
    xaxis_title='Month',
    yaxis_title='Return (%)',
    template='plotly_white',
    height=400,
    xaxis_tickangle=-45
)

fig_monthly.show()

print("\nMonthly Performance Summary:")
print("="*50)
print(f"Average Monthly Return: {df_monthly['Monthly_Return'].mean():.2f}%")
print(f"Best Month: {df_monthly['Monthly_Return'].max():.2f}%")
print(f"Worst Month: {df_monthly['Monthly_Return'].min():.2f}%")
print(f"Positive Months: {(df_monthly['Monthly_Return'] > 0).sum()} out of {len(df_monthly)}")

## 10. Insights and Interpretation

In [ ]:
# Generate insights
current_rsi = df['RSI'].iloc[-1]
latest_close = df['Close'].iloc[-1]
sma_value = df['SMA_20'].iloc[-1]

# RSI interpretation
if current_rsi > 70:
    rsi_signal = "Overbought - Potential sell signal"
elif current_rsi < 30:
    rsi_signal = "Oversold - Potential buy signal"
else:
    rsi_signal = "Neutral"

# Trend analysis
if latest_close > sma_value:
    trend = "Upward (Price above SMA20)"
else:
    trend = "Downward (Price below SMA20)"

print("ANALYSIS SUMMARY")
print("="*50)
print(f"\n1. PRICE PERFORMANCE:")
print(f"   - Current Price: ${latest_close:.2f}")
print(f"   - Period Return: {period_return:.2f}%")
print(f"   - Price Range: ${lowest_price:.2f} - ${highest_price:.2f}")

print(f"\n2. TECHNICAL INDICATORS:")
print(f"   - RSI (14): {current_rsi:.1f} ({rsi_signal})")
print(f"   - SMA (20): ${sma_value:.2f}")
print(f"   - Current Trend: {trend}")

print(f"\n3. RISK METRICS:")
print(f"   - Annualized Volatility: {volatility_annual:.2f}%")
print(f"   - Average Daily Volume: {avg_volume:,.0f} shares")

print(f"\n4. MONTHLY INSIGHTS:")
print(f"   - Average Monthly Return: {df_monthly['Monthly_Return'].mean():.2f}%")
print(f"   - Win Rate: {(df_monthly['Monthly_Return'] > 0).mean()*100:.1f}%")

## 11. Export Processed Data

In [ ]:
# Save processed data to CSV
output_file = f'{stock_symbol}_analysis_data.csv'
df_export = df[['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'EMA_20', 'RSI', 'Daily_Return']].copy()
df_export.index = df_export.index.strftime('%Y-%m-%d')
df_export.to_csv(output_file)

print(f"Data exported to: {output_file}")
print(f"\nExport preview:")
df_export.tail(10)

## 12. Conclusion

This notebook demonstrates the complete analytical workflow for the Stock Financial Analysis Tool using iFinD data source:

1. **Problem Definition:** Created an interactive tool for stock analysis targeting students and individual investors
2. **Data Acquisition:** Successfully fetched historical stock data from iFinD Financial Database
3. **Data Cleaning:** Validated data quality with no missing values
4. **Feature Engineering:** Calculated key technical indicators (SMA, EMA, RSI, Volatility)
5. **Exploratory Analysis:** Visualized price trends, volume, and returns distribution
6. **Insights Generation:** Provided actionable analysis based on technical indicators

### Data Source Information:
- **Source:** iFinD Financial Database
- **Access Date:** April 16, 2025
- **Coverage:** US stocks, Hong Kong stocks, China A-shares

The interactive Streamlit application built on this workflow allows users to:
- Analyze any stock symbol from supported markets
- Customize date ranges and technical indicators
- Visualize data through interactive charts
- Download analysis results

**Note:** This analysis is for educational purposes only and should not be considered as investment advice.